In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import joblib
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import warnings
import os
import sys

# Import XGBoost Regressor
import xgboost as xgb

# Suppress all warnings
warnings.filterwarnings("ignore")

# Define the directory for artifacts
ARTIFACTS_DIR = 'Scenario 4'

def prepare_data(df):
    """
    Performs initial data preparation and feature engineering on the raw sales data.
    Note: Customer count related features are removed from being model inputs,
    but we still need customer count to store as a baseline for projections.
    """
    df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', errors='coerce')
    df = df.dropna(subset=['Order Date'])
    df['Month_Year'] = df['Order Date'].dt.to_period('M')
    
    # Calculate monthly customer count for historical baseline, but not for model features
    monthly_customers = df.groupby('Month_Year')['Customer ID'].nunique().reset_index()
    monthly_customers.rename(columns={'Customer ID': 'Customer_Count_Monthly'}, inplace=True)

    # Aggregate total sales by Product Name, Product ID, Category, Sub-Category and Month_Year
    monthly_product_sales = df.groupby(['Month_Year', 'Product Name', 'Product ID', 'Category', 'Sub-Category'])['Sales'].sum().reset_index()

    # Merge monthly customer count for baseline extraction later, not for model features
    df_prepared = pd.merge(monthly_product_sales, monthly_customers, on='Month_Year', how='left')

    df_prepared['Month_Year_dt'] = df_prepared['Month_Year'].dt.to_timestamp()
    df_prepared['Year'] = df_prepared['Month_Year_dt'].dt.year
    df_prepared['Month'] = df_prepared['Month_Year_dt'].dt.month
    df_prepared['Quarter'] = df_prepared['Month_Year_dt'].dt.quarter

    unique_months = sorted(df_prepared['Month_Year_dt'].unique())
    time_index_map = {date: i for i, date in enumerate(unique_months)}
    df_prepared['Time_Index'] = df_prepared['Month_Year_dt'].map(time_index_map)

    df_prepared['Month_sin'] = np.sin(2 * np.pi * df_prepared['Month'] / 12)
    df_prepared['Month_cos'] = np.cos(2 * np.pi * df_prepared['Month'] / 12)
    
    df_prepared = df_prepared.sort_values(by=['Product Name', 'Month_Year'])

    # Removed: Customer_Count_Prev_Month and Customer_Increase_Pct_Historical calculations
    # because they were primarily for features that are now removed.

    print("Data preparation complete. Sample of prepared data with all new features:")
    print(df_prepared.head())
    return df_prepared

def encode_data(df_prepared):
    """
    Encodes categorical features in the prepared DataFrame using LabelEncoder.
    """
    product_encoder = LabelEncoder()
    category_encoder = LabelEncoder()
    subcategory_encoder = LabelEncoder()

    df_encoded = df_prepared.copy()
    df_encoded['Product_Name_Encoded'] = product_encoder.fit_transform(df_encoded['Product Name'])
    df_encoded['Category_Encoded'] = category_encoder.fit_transform(df_encoded['Category'])
    df_encoded['SubCategory_Encoded'] = subcategory_encoder.fit_transform(df_encoded['Sub-Category'])

    print("\nEncoding complete. Sample of encoded data:")
    print(df_encoded.head())
    return df_encoded, product_encoder, category_encoder, subcategory_encoder

def build_model():
    """
    Builds and returns an XGBoost Regressor model.
    """
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    print("\nModel (XGBRegressor) built successfully.")
    return model

def train_model(df_encoded, model):
    """
    Trains the sales prediction model with updated features.
    """
    features = [
        'Year', 'Month', 'Quarter', 'Time_Index', 'Month_sin', 'Month_cos',
        'Product_Name_Encoded', 'Category_Encoded', 'SubCategory_Encoded'
        # Customer_Count_Monthly is still explicitly excluded from model features
    ]
    target = 'Sales'

    X = df_encoded[features]
    y = df_encoded[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"\nTraining model with features: {features} and target: {target}")
    print(f"Training data shape: {X_train.shape}, Test data shape: {X_test.shape}")

    model.fit(X_train, y_train)
    print("Model training complete.")
    return model, X_test, y_test

def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series):
    """
    Evaluates the trained model's performance on the test set using various regression metrics.
    """
    print("Evaluating model...")
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("\n--- Model Evaluation Report (Regression) ---")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Mean Squared Error (MSE): {mse:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"R-squared (R2): {r2:.4f}")
    print("-------------------------------------------\n")

def save_artifacts(model, product_encoder, category_encoder, subcategory_encoder, 
                   last_historical_time_index, last_month_customer_count, product_info_map, 
                   filepath_model='sales_prediction_model.joblib',
                   filepath_product_encoder='product_encoder.joblib',
                   filepath_category_encoder='category_encoder.joblib',
                   filepath_subcategory_encoder='subcategory_encoder.joblib',
                   filepath_prediction_context='prediction_context.joblib'):
    """
    Saves the trained model and all necessary encoders and prediction context to disk.
    """
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)
    
    model_path = os.path.join(ARTIFACTS_DIR, filepath_model)
    product_encoder_path = os.path.join(ARTIFACTS_DIR, filepath_product_encoder)
    category_encoder_path = os.path.join(ARTIFACTS_DIR, filepath_category_encoder)
    subcategory_encoder_path = os.path.join(ARTIFACTS_DIR, filepath_subcategory_encoder)
    prediction_context_path = os.path.join(ARTIFACTS_DIR, filepath_prediction_context)

    joblib.dump(model, model_path)
    joblib.dump(product_encoder, product_encoder_path)
    joblib.dump(category_encoder, category_encoder_path)
    joblib.dump(subcategory_encoder, subcategory_encoder_path)
    
    # Save the prediction context as a dictionary
    prediction_context = {
        'last_historical_time_index': last_historical_time_index,
        'last_month_customer_count': last_month_customer_count, 
        'product_info_map': product_info_map
    }
    joblib.dump(prediction_context, prediction_context_path)
    
    print(f"\nModel saved to {model_path}")
    print(f"Product encoder saved to {product_encoder_path}")
    print(f"Category encoder saved to {category_encoder_path}")
    print(f"Sub-Category encoder saved to {subcategory_encoder_path}")
    print(f"Prediction context saved to {prediction_context_path}")


def load_artifacts(filepath_model='sales_prediction_model.joblib',
                   filepath_product_encoder='product_encoder.joblib',
                   filepath_category_encoder='category_encoder.joblib',
                   filepath_subcategory_encoder='subcategory_encoder.joblib',
                   filepath_prediction_context='prediction_context.joblib'):
    """
    Loads a previously trained model, encoders, and prediction context from disk.
    """
    model_path = os.path.join(ARTIFACTS_DIR, filepath_model)
    product_encoder_path = os.path.join(ARTIFACTS_DIR, filepath_product_encoder)
    category_encoder_path = os.path.join(ARTIFACTS_DIR, filepath_category_encoder)
    subcategory_encoder_path = os.path.join(ARTIFACTS_DIR, filepath_subcategory_encoder)
    prediction_context_path = os.path.join(ARTIFACTS_DIR, filepath_prediction_context)

    if not all(os.path.exists(p) for p in [model_path, product_encoder_path, category_encoder_path, subcategory_encoder_path, prediction_context_path]):
        raise FileNotFoundError(f"One or more artifacts not found in {ARTIFACTS_DIR}. Please ensure the model has been trained and saved.")

    model = joblib.load(model_path)
    product_encoder = joblib.load(product_encoder_path) 
    category_encoder = joblib.load(category_encoder_path) 
    subcategory_encoder = joblib.load(subcategory_encoder_path) 
    
    prediction_context = joblib.load(prediction_context_path)
    last_historical_time_index = prediction_context['last_historical_time_index']
    last_month_customer_count = prediction_context['last_month_customer_count'] 
    product_info_map = prediction_context['product_info_map']
    
    print(f"\nModel loaded from {model_path}")
    print(f"Encoders loaded.")
    print(f"Prediction context loaded from {prediction_context_path}")
    
    return model, product_encoder, category_encoder, subcategory_encoder, last_historical_time_index, last_month_customer_count, product_info_map

def make_predictions(prediction_input: dict, output_format='dataframe'):
    """
    Generates sales predictions for each product for the next month(s) based on
    time and product-specific features. Customer count projections are now calculated
    and included in the output, but not as a model input.
    This function now loads artifacts internally.

    Args:
        prediction_input (dict): Dictionary containing "percentage_increase" and "Forecast Period".
        output_format (str): Desired output format. Can be 'dataframe' or 'json'.

    Returns:
        pd.DataFrame or dict: Sales predictions in the specified format.
    """
    # Load artifacts internally
    model, product_encoder, category_encoder, subcategory_encoder, \
    last_historical_time_index, last_month_customer_count, product_info_map = load_artifacts()

    percentage_increase = prediction_input["percentage_increase"]
    forecast_period = prediction_input["Forecast Period"]

    forecast_results = {} 
    all_predictions_list = [] 

    today_for_forecast_start = datetime.now()
    
    current_projected_customer_count = last_month_customer_count 

    print(f"\nStarting forecast from {today_for_forecast_start.strftime('%Y-%m-%d')} based on last historical customer count of {last_month_customer_count:.2f}")

    for i in range(forecast_period):
        target_date = today_for_forecast_start + relativedelta(months=i)
        
        forecast_month_str = target_date.strftime('%Y-%m')

        if isinstance(percentage_increase, (int, float)):
            current_pct_increase = percentage_increase
        elif isinstance(percentage_increase, list):
            if i < len(percentage_increase):
                current_pct_increase = percentage_increase[i]
            else:
                current_pct_increase = percentage_increase[-1]
        else:
            raise ValueError("Invalid format for 'percentage_increase'. Must be a single number or a list.")

        current_projected_customer_count *= (1 + current_pct_increase / 100) 

        forecast_time_index = last_historical_time_index + (i + 1)
        
        forecast_quarter = (target_date.month - 1) // 3 + 1

        forecast_month_sin = np.sin(2 * np.pi * target_date.month / 12)
        forecast_month_cos = np.cos(2 * np.pi * target_date.month / 12)

        month_predictions = {}
        for product_name, p_info in product_info_map.items():
            prediction_df = pd.DataFrame([{
                'Year': target_date.year,
                'Month': target_date.month,
                'Quarter': forecast_quarter,
                'Time_Index': forecast_time_index,
                'Month_sin': forecast_month_sin,
                'Month_cos': forecast_month_cos,
                'Product_Name_Encoded': p_info['Product_Name_Encoded'],
                'Category_Encoded': p_info['Category_Encoded'],
                'SubCategory_Encoded': p_info['SubCategory_Encoded']
                # Customer_Count_Monthly is NOT included here as a feature for the model
            }])
            
            predicted_sales = model.predict(prediction_df)[0]
            rounded_predicted_sales = max(0, round(predicted_sales))
            
            month_predictions[product_name] = rounded_predicted_sales

            all_predictions_list.append({
                'Forecast_Month': forecast_month_str,
                'Product_Name': product_name,
                'Predicted_Sales': rounded_predicted_sales,
                'Projected_Customer_Count': round(current_projected_customer_count, 2) 
            })

        forecast_results[forecast_month_str] = month_predictions

    print("\nSales predictions generated:")
    
    if output_format == 'dataframe':
        df_predictions = pd.DataFrame(all_predictions_list)
        # print(df_predictions.to_string())
        return df_predictions
    elif output_format == 'json':
        # print(forecast_results)
        return forecast_results
    else:
        raise ValueError("Invalid output_format. Choose 'dataframe' or 'json'.")



In [26]:

if __name__ == "__main__":
    # --- Dummy Data Creation (Replace with your actual data loading) ---
    
    df = pd.read_csv(f'{os.getcwd()}/../data/raw_data/Complete.csv')
    # df = pd.read_csv(f'{os.getcwd()}/../data/raw_data/Sample - Superstore.csv', encoding='windows-1254')

    print("Starting the sales prediction workflow...\n")

    # --- Workflow Execution ---

    # 1. Prepare Data
    df_prepared = prepare_data(df.copy())

    # 2. Encode Categorical Features
    df_encoded, product_encoder, category_encoder, subcategory_encoder = encode_data(df_prepared)

    # 3. Build the Model
    model = build_model()

    # 4. Train the Model
    trained_model, X_test, y_test = train_model(df_encoded, model)

    # 5. Evaluate the Model
    evaluate_model(trained_model, X_test, y_test)

    # --- Extract necessary info from df_encoded for saving as prediction context ---
    last_historical_month_dt = df_encoded['Month_Year_dt'].max()
    last_historical_time_index = df_encoded[df_encoded['Month_Year_dt'] == last_historical_month_dt]['Time_Index'].iloc[0]
    
    last_month_customer_count = df_encoded[
        df_encoded['Month_Year_dt'] == last_historical_month_dt
    ]['Customer_Count_Monthly'].mean()


    unique_products_df = df_encoded[
        ['Product Name', 'Product_Name_Encoded', 'Category', 'Category_Encoded', 'Sub-Category', 'SubCategory_Encoded']
    ].drop_duplicates().reset_index(drop=True)
    
    product_info_map = {
        row['Product Name']: {
            'Product_Name_Encoded': row['Product_Name_Encoded'],
            'Category_Encoded': row['Category_Encoded'],
            'SubCategory_Encoded': row['SubCategory_Encoded']
        }
        for _, row in unique_products_df.iterrows()
    }
    # --- End extraction for saving ---

    # 6. Save Artifacts 
    save_artifacts(trained_model, product_encoder, category_encoder, subcategory_encoder,
                   last_historical_time_index, last_month_customer_count, product_info_map)

    # --- From this point onwards, df_encoded is no longer needed ---

    # # 7. Load Artifacts 
    # loaded_model, loaded_product_encoder, loaded_category_encoder, loaded_subcategory_encoder, \
    # loaded_last_historical_time_index, loaded_last_month_customer_count, loaded_product_info_map = load_artifacts()

    # 8. Make Predictions
    print("\n--- Initiating Sales Predictions from Today ---")

    prediction_input_1 = {
        "percentage_increase": 5, 
        "Forecast Period": 3 
    }
    print("\nPrediction with constant 5% monthly customer increase (DataFrame output):")
    forecast_df = make_predictions(prediction_input_1, output_format='dataframe')
    print("\nType of output for forecast_df:", type(forecast_df))


    # prediction_input_2 = {
    #     "percentage_increase": [5, 7, 10], 
    #     "Forecast Period": 3 
    # }
    # print("\nPrediction with varied monthly customer increases [5%, 7%, 10%] (JSON output):")
    # forecast_json = make_predictions(loaded_model, loaded_last_historical_time_index, 
    #                                  loaded_last_month_customer_count, loaded_product_info_map, 
    #                                  prediction_input_2, output_format='json')
    # print("\nType of output for forecast_json:", type(forecast_json))

    # prediction_input_3 = {
    #     "percentage_increase": 8, 
    #     "Forecast Period": 1 
    # }
    # print("\nPrediction for a single month with 8% customer increase (Default DataFrame output):")
    # forecast_df_single = make_predictions(loaded_model, loaded_last_historical_time_index, 
    #                                       loaded_last_month_customer_count, loaded_product_info_map, 
    #                                       prediction_input_3)
    # print("\nType of output for forecast_df_single:", type(forecast_df_single))


Starting the sales prediction workflow...

Data preparation complete. Sample of prepared data with all new features:
      Month_Year                                       Product Name  \
21117    2017-09  "While you Were Out" Message Book, One Form pe...   
21800    2017-10  "While you Were Out" Message Book, One Form pe...   
22376    2017-11  "While you Were Out" Message Book, One Form pe...   
9951     2015-11           #10 Gummed Flap White Envelopes, 100/Box   
11137    2016-01           #10 Gummed Flap White Envelopes, 100/Box   

            Product ID         Category Sub-Category   Sales  \
21117  OFF-PA-10003424  Office Supplies        Paper   8.904   
21800  OFF-PA-10003424  Office Supplies        Paper   7.420   
22376  OFF-PA-10003424  Office Supplies        Paper   8.904   
9951   OFF-EN-10001137  Office Supplies    Envelopes   6.608   
11137  OFF-EN-10001137  Office Supplies    Envelopes  16.520   

       Customer_Count_Monthly Month_Year_dt  Year  Month  Quarter  Time

In [24]:
forecast_df


,Forecast_Month,Product_Name,Predicted_Sales,Projected_Customer_Count
0,2025-07,"""While you Were Out"" Message Book, One Form pe...",1118,220.5
1,2025-07,"#10 Gummed Flap White Envelopes, 100/Box",1089,220.5
2,2025-07,#10 Self-Seal White Envelopes,1089,220.5
3,2025-07,"#10 White Business Envelopes,4 1/8 x 9 1/2",1089,220.5
4,2025-07,"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",1089,220.5
...,...,...,...,...
5599,2025-09,iOttie HLCRIO102 Car Mount,966,243.1
5600,2025-09,iOttie XL Car Mount,966,243.1
5601,2025-09,iPhone 14,1083,243.1
5602,2025-09,invisibleSHIELD by ZAGG Smudge-Free Screen Pro...,1083,243.1
